In [ ]:
import hashlib
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf, expr
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("Day04_DeID").getOrCreate()

def simulate_fpe(value: str) -> str:
    if not value: return value
    return "FPE-" + hashlib.sha256(value.encode()).hexdigest()[:12]

fpe_udf = udf(simulate_fpe, StringType())

# Create synthetic raw patient dataframe
data = [
    ("p-001", "123-45-6789", "MRN-101", "1980-05-15", "Patient notes with 123-45-6789 here."),
    ("p-002", "987-65-4321", "MRN-202", "1992-11-20", "No PII in these notes.")
]
columns = ["payload_id", "ssn", "mrn", "birth_date", "clinical_notes"]
raw_df = spark.createDataFrame(data, columns)

print("RAW DATAFRAME:")
raw_df.show(truncate=False)

# Apply De-ID Rules
deid_df = raw_df \
    .withColumn("ssn", fpe_udf(col("ssn"))) \
    .withColumn("mrn", fpe_udf(col("mrn"))) \
    .withColumn("birth_date", expr("date_add(birth_date, cast(conv(substr(md5(payload_id), 1, 8), 16, 10) % 60 - 30 as int))")) \
    .withColumn("clinical_notes", expr("regexp_replace(clinical_notes, '\\\\b\\\\d{3}-\\\\d{2}-\\\\d{4}\\\\b', '[REDACTED_SSN]')"))

print("DE-IDENTIFIED DATAFRAME:")
deid_df.show(truncate=False)
